# Concept segmentation in images with SAM 3


Say what you are looking for, and SAM 3 returns **every instance of it** in the image —
mask, box and score each. No clicking required, and no fixed label set: the concept is a
phrase you write.

That is the difference from SAM 2, which segments *the thing you pointed at* and has no idea
what it is. Here you never point unless you want to.

| you want | prompt | what comes back |
|---|---|---|
| every instance of a described thing | `ConceptPrompt(text="wheel")` | one mask per instance, keyed by score |
| the same, but biased toward one example | `geometry=GeometryPrompt.concept_box(xyxy)` | same instances, the boxed one scored up |
| the same, biased *away* from one | `GeometryPrompt.concept_box(xyxy, label=0)` | the boxed instance scored down |
| the same, from a click | `GeometryPrompt.click(1, (x, y))` | same instances, re-scored around the point |
| "is this concept here at all?" | any of the above | `result.presence`, 0..1, before you look at instances |

This notebook covers text prompting, what `confidence_threshold` actually does, presence,
box and click exemplars, and the SAM 3.1 multiplex predictor — which is the same call.

For the same model over a video, tracking each instance across frames, see
[`sam3_video_predictor_example.ipynb`](./sam3_video_predictor_example.ipynb).


## Set-up

SAM 3 weights are access-gated — request them via [Meta AI](https://ai.meta.com/sam), then
`pixi run download-sam3` and `pixi run download-sam3-1`. SAM 3 does its preprocessing on the
GPU, so a CUDA device is required rather than merely recommended.


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

if os.path.isdir("notebooks"):     # started from the repo root rather than notebooks/
    sys.path.insert(0, "notebooks")
import nb_utils as nb  # shared plotting helpers — see notebooks/nb_utils.py

nb.use_repo_root()     # so "configs/..." and "checkpoints/..." resolve
nb.use_dark(False)     # light figures; see nb_utils.use_dark for PyCharm's inversion
device = nb.pick_device(require_cuda=True)

nb.require_checkpoints(("checkpoints/sam3.pt", "pixi run download-sam3"))


In [ ]:
from sam.build_sam import build_sam3, build_sam3_multiplex
from sam.prompts import ConceptPrompt, GeometryPrompt

predictor = build_sam3("configs/sam3/sam3.yaml", "./checkpoints/sam3.pt", device=device)
print("ready")


Two images, and one helper that draws whatever `predict` returns.


In [ ]:
TRUCK = np.array(Image.open("notebooks/images/truck.jpg").convert("RGB"))
GROCERIES = np.array(Image.open("notebooks/images/groceries.jpg").convert("RGB"))
print("truck:", TRUCK.shape, " groceries:", GROCERIES.shape)


def show(image, result, title, extra=None, width=9.0):
    """Draw every detected instance: mask, box, and the score above it."""
    masks = (result.masks_logits > 0).cpu().numpy()
    boxes = result.boxes.float().cpu().numpy()
    scores = result.scores.float().cpu().numpy()

    fig, ax = plt.subplots(figsize=(width, width * image.shape[0] / image.shape[1]))
    ax.imshow(image)
    ax.axis("off")
    ax.set_title(f"{title}\n{len(scores)} instance(s) · presence {result.presence:.3f}")
    for i, (mask, box, score) in enumerate(zip(masks, boxes, scores)):
        nb.show_mask(mask, ax, obj_id=i)
        nb.show_box(box, ax, obj_id=i, label=f"{score:.2f}")
    if extra is not None:
        extra(ax)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


## 1. Text — every instance of a concept

One call. The phrase is the whole prompt.


In [ ]:
result = predictor.predict(GROCERIES, ConceptPrompt(text="paper bag"))

print("masks_logits:", tuple(result.masks_logits.shape), "  (N, H, W) — > 0 is foreground")
print("boxes:       ", tuple(result.boxes.shape), "     — xyxy pixels")
print("scores:      ", result.scores.float().cpu().numpy().round(3))
print("presence:    ", round(result.presence, 3))
print("instance_ids:", result.instance_ids.cpu().numpy())

show(GROCERIES, result, 'ConceptPrompt(text="paper bag")')


Every field is per-instance except `presence`, which is one number for the image.

`masks_logits` are *logits*: positive means foreground, so `result.masks_logits > 0` is your
binary mask. The boxes are the detector's own, in pixels — you do not derive them from the
masks the way the video notebooks have to.

The concept is a phrase, not a class, so the wording is yours to choose. `"grocery bag"`
finds the same four bags here; a phrase the image does not contain finds nothing at all,
which is section 3.


In [ ]:
result = predictor.predict(TRUCK, ConceptPrompt(text="wheel"))
show(TRUCK, result, 'ConceptPrompt(text="wheel")')


Four, not two: the two far-side wheels are visible under the body, and each is its own
instance with its own score. Nothing told the model how many to expect.


## 2. `confidence_threshold` — how sure is sure enough

`predict` keeps instances whose presence-weighted score clears the threshold. The default,
0.5, is not a magic number; it is a dial, and the useful range depends on the image.


In [ ]:
for t in (0.1, 0.3, 0.5, 0.7, 0.9):
    r = predictor.predict(GROCERIES, ConceptPrompt(text="paper bag"), confidence_threshold=t)
    print(f"threshold {t}: {int(r.boxes.shape[0]):2} instance(s)  "
          f"scores={r.scores.float().cpu().numpy().round(2)}")


The four real bags sit around 0.89–0.90, so anything from 0.3 to 0.7 returns exactly
them. Drop to 0.1 and you also collect a tail of ~0.1-scored guesses; raise to 0.9 and you
keep only the single best. A wide plateau like this one means the model is confident — a
threshold that changes the count continuously would mean the opposite.


In [ ]:
loose = predictor.predict(GROCERIES, ConceptPrompt(text="paper bag"), confidence_threshold=0.1)
show(GROCERIES, loose, "confidence_threshold=0.1 — the tail is visible")


## 3. Presence — is the concept in this image at all

`presence` answers that before you look at a single instance, which is what you want when
the answer is often "no".


In [ ]:
for name, image in (("groceries", GROCERIES), ("truck", TRUCK)):
    r = predictor.predict(image, ConceptPrompt(text="elephant"))
    print(f"{name:10} 'elephant': {int(r.boxes.shape[0])} instance(s), "
          f"presence {r.presence:.3f}")


Zero instances and a presence of essentially zero — the model is not hedging. Compare
with 0.99 for `"wheel"` on the truck. Use `presence` as the gate and `scores` to rank what
survives it.


## 4. Box exemplar — point at one, keep the concept

Pass a `GeometryPrompt` as `geometry` and the box becomes an *example* of what you mean, not
a selection. The concept still drives detection; the box biases it.


In [ ]:
REAR_WHEEL = (450.0, 620.0, 670.0, 840.0)  # xyxy pixels, the rear wheel

plain = predictor.predict(TRUCK, ConceptPrompt(text="wheel"))
hinted = predictor.predict(
    TRUCK, ConceptPrompt(text="wheel"),
    geometry=GeometryPrompt.concept_box(REAR_WHEEL),
)

print("no exemplar:", plain.scores.float().cpu().numpy().round(3),
      f"presence {plain.presence:.3f}")
print("box exemplar:", hinted.scores.float().cpu().numpy().round(3),
      f"presence {hinted.presence:.3f}")

show(TRUCK, hinted, "wheel + positive box exemplar (dashed)",
     extra=lambda ax: nb.show_box(REAR_WHEEL, ax, label="exemplar", style="--"))


Still four wheels — the exemplar did not narrow the search to one object. What moved is
confidence: the boxed wheel scores higher than it did unprompted, and presence goes to 1.00.
That is the contract. If you want *only* that object, you want the tracker box
(`GeometryPrompt.box(obj_id, xyxy)`) or SAM 2, not a concept prompt.

`label=0` inverts the bias — "everything matching the concept **except** this one".


In [ ]:
against = predictor.predict(
    TRUCK, ConceptPrompt(text="wheel"),
    geometry=GeometryPrompt.concept_box(REAR_WHEEL, label=0),
)
print("negative exemplar:", against.scores.float().cpu().numpy().round(3))

show(TRUCK, against, "wheel + negative box exemplar (dashed)",
     extra=lambda ax: nb.show_box(REAR_WHEEL, ax, label="not this", style="--"))


The boxed wheel drops well below the others rather than disappearing. A negative
exemplar reweights; it does not delete. Threshold accordingly if you need it gone.


## 5. Click exemplar

Same idea, one point instead of four numbers. `click` takes an `obj_id` because the video
API needs one to key its tracklets — on the image path nothing reads it, so any value does.


In [ ]:
CLICK = (560.0, 730.0)  # centre of the rear wheel

clicked = predictor.predict(
    TRUCK, ConceptPrompt(text="wheel"),
    geometry=GeometryPrompt.click(1, CLICK),
)
print("click exemplar:", clicked.scores.float().cpu().numpy().round(3),
      f"presence {clicked.presence:.3f}")

show(TRUCK, clicked, "wheel + click exemplar",
     extra=lambda ax: nb.show_points([CLICK], [1], ax))


A click moves the scores too, but not the way the box did: here every wheel comes back a
little *lower*, the clicked one included (0.95 unprompted against 0.89 with the click). A box
says where **and** how big; a lone point says less, and this detector was already sure. Reach
for the box when you have one.


## 6. SAM 3.1 (multiplex) — the same call

SAM 3.1 is a different checkpoint and a different config; the calling code is unchanged. Its
scoring differs internally — the score is the joint logit with presence already folded in,
and the boxes are derived from the masks rather than predicted directly — so expect the same
instances with slightly different numbers.

The two checkpoints are several GB each and do not comfortably co-reside, so free the first.


In [ ]:
nb.free(globals(), "predictor")

nb.require_checkpoints(("checkpoints/sam3.1_multiplex.pt", "pixi run download-sam3-1"))
mux = build_sam3_multiplex(
    "configs/sam3/sam3.1.yaml", "./checkpoints/sam3.1_multiplex.pt", device=device,
)

mux_result = mux.predict(TRUCK, ConceptPrompt(text="wheel"))
print("SAM 3.1 scores:", mux_result.scores.float().cpu().numpy().round(3),
      f"presence {mux_result.presence:.3f}")

show(TRUCK, mux_result, "SAM 3.1 multiplex — same prompt, same API")


Same four wheels, boxes within a pixel or two of the base model's. On images the
multiplex checkpoint mostly buys you consistency with the video predictor that tracks up to
16 objects in one joint forward pass; the image call is identical either way.


## What you can do from here

| | how |
|---|---|
| Find every instance of a thing | `predictor.predict(image, ConceptPrompt(text="..."))` |
| Tighten or loosen the result set | `confidence_threshold=` (0.5 default) |
| Ask whether it is there at all | `result.presence` |
| Bias toward an example | `geometry=GeometryPrompt.concept_box((x0, y0, x1, y1))` |
| Bias away from one | the same, `label=0` |
| Bias from a click | `geometry=GeometryPrompt.click(1, (x, y))` |
| Binary masks | `result.masks_logits > 0` |
| Run SAM 3.1 instead | `build_sam3_multiplex` + `sam3.1.yaml` + `sam3.1_multiplex.pt` |
| Track those instances across a video | [`sam3_video_predictor_example.ipynb`](./sam3_video_predictor_example.ipynb) |

Mask geometry (`GeometryPrompt` with `masks_logits`) raises `NotImplementedError`: neither
checkpoint ships `mask_encoder` weights. Use a box or a point.
